# Cement policy: sector and year fixed effects

Estimate post-2010 changes in NOₓ, SO₂, and PM₂.₅ emissions for sector 7 using sector and year fixed effects, log emissions (2005–2023), and state-clustered standard errors. Controls include other component sectors, with an alternative excluding electric utilities; aggregate sectors 0 and 99 are excluded throughout. Sector 7 represents “Other Industrial Processes,” not cement alone. Data come from `NEI_econ230a.csv`; missing values are not imputed, and log models exclude nonpositive emissions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

YEARS = [2005, 2008, 2011, 2014, 2017, 2020, 2023]
LOG_OUTCOME = True
POLLUTANTS = ["NOX", "SO2", "PM25"]
OUTPUT = Path("cement_policy_regressions.tex")
df = pd.read_csv("NEI_econ230a.csv")
for column in ["state", "pollutant", "sector"]:
    df[column] = df[column].str.strip()
states = set("AL AK AZ AR CA CO CT DE DC FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY".split())
df = df.loc[df["state"].isin(states) & df["year"].isin(YEARS)
            & df["pollutant"].isin(POLLUTANTS) & ~df["sector_code"].isin([0, 99])].copy()
assert not df.duplicated(["state", "sector_code", "year", "pollutant"]).any()
assert df[["state", "sector_code", "year", "pollutant"]].notna().all().all()
assert set(df["year"]) == set(YEARS)
assert df["emissions_tons"].dropna().ge(0).all()
display(df[["sector_code", "sector"]].drop_duplicates().sort_values("sector_code").reset_index(drop=True))

valid = np.isfinite(df["emissions_tons"])
if LOG_OUTCOME:
    valid &= df["emissions_tons"].gt(0)
audit = df.assign(eligible=valid).groupby(["pollutant", "sector_code"]).agg(
    observed_rows=("year", "size"),
    missing_emissions=("emissions_tons", lambda x: x.isna().sum()),
    zero_emissions=("emissions_tons", lambda x: x.eq(0).sum()),
    eligible_rows=("eligible", "sum"),
)
audit["absent_rows"] = len(states) * len(YEARS) - audit["observed_rows"]
display(audit)
sample = df.loc[valid].copy()
sample["outcome"] = np.log(sample["emissions_tons"]) if LOG_OUTCOME else sample["emissions_tons"]
# Post-policy period: 2011 and later (policy effective November 2010).
sample["post"] = sample["year"].ge(2011).astype(int)
sample["treated_post"] = sample["sector_code"].eq(7).astype(int) * sample["post"]
assert set(sample["post"]) == {0, 1}
print("Outcome:", "Natural log emissions" if LOG_OUTCOME else "Emissions in tons")
print("Years:", YEARS)

## Estimate both specifications

The formula is `outcome ~ treated_post + C(sector_code) + C(year)`. A separate treated-sector indicator and post indicator are absorbed by sector and year fixed effects. The alternative uses the identical eligible sample except for dropping sector 1. All eligible treated observations are retained in both specifications.

In [ ]:
specifications = {"Main": False, "Excluding utilities": True}
models, records = {}, []
for pollutant in POLLUTANTS:
    pollutant_sample = sample.loc[sample["pollutant"].eq(pollutant)]
    for specification, exclude_utilities in specifications.items():
        group = pollutant_sample.loc[pollutant_sample["sector_code"].ne(1)] if exclude_utilities else pollutant_sample
        assert set(group.loc[group["sector_code"].eq(7), "post"]) == {0, 1}
        model = smf.ols("outcome ~ treated_post + C(sector_code) + C(year)",
                        data=group, missing="raise").fit(
            cov_type="cluster",
            cov_kwds={"groups": group["state"], "use_correction": True, "df_correction": True},
            use_t=True,
        )
        assert np.linalg.matrix_rank(model.model.exog) == model.model.exog.shape[1]
        assert int(model.nobs) == len(group)
        models[(pollutant, specification)] = model
        records.append({"pollutant": pollutant, "specification": specification,
                        "beta": model.params["treated_post"], "se": model.bse["treated_post"],
                        "p_value": model.pvalues["treated_post"], "n": int(model.nobs),
                        "states": group["state"].nunique(), "sectors": group["sector_code"].nunique(),
                        "treated_n": int(group["sector_code"].eq(7).sum()), "r_squared": model.rsquared})
results = pd.DataFrame(records).set_index(["pollutant", "specification"])
assert results.groupby(level="pollutant")["treated_n"].nunique().eq(1).all()
assert np.isfinite(results).all().all()

In [ ]:
def stars(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""

rows, index = [], []
for pollutant in POLLUTANTS:
    panel = results.loc[pollutant].reindex(specifications)
    entries = {
        "Sector 7 × Post": [f"{r.beta:.3f}{stars(r.p_value)}" for r in panel.itertuples()],
        "State-clustered SE": [f"({r.se:.3f})" for r in panel.itertuples()],
        "p-value": [f"{p:.4g}" for p in panel["p_value"]],
        "Observations": [f"{int(n):,}" for n in panel["n"]],
        "Treated observations": [f"{int(n):,}" for n in panel["treated_n"]],
        "State clusters": [str(int(n)) for n in panel["states"]],
        "Sectors": [str(int(n)) for n in panel["sectors"]],
        "R² (overall)": [f"{r:.3f}" for r in panel["r_squared"]],
        "Sector FE": ["Yes", "Yes"], "Year FE": ["Yes", "Yes"],
    }
    for statistic, values in entries.items():
        index.append((pollutant, statistic))
        rows.append(values)
regression_table = pd.DataFrame(rows, columns=list(specifications),
    index=pd.MultiIndex.from_tuples(index, names=["Pollutant", "Statistic"]))
display(regression_table)
print("* p<0.10, ** p<0.05, *** p<0.01. State-clustered standard errors in parentheses.")
if LOG_OUTCOME:
    print("Beta is a log-point difference-in-differences; 100 × (exp(beta) − 1) gives its percentage interpretation.")

## Export the table

Add `\usepackage{booktabs}` to your LaTeX preamble and `\input{cement_policy_regressions.tex}` where the table should appear. The CSV export retains unrounded coefficients, standard errors, and p-values.

In [ ]:
labels = {"NOX": r"NO$_x$", "SO2": r"SO$_2$", "PM25": r"PM$_{2.5}$"}
lines = [r"\begin{table}[htbp]", r"\centering", r"\small",
         r"\caption{Sector 7 emissions after the 2010 cement rule}",
         r"\label{tab:cement-policy}", r"\begin{tabular}{lcc}", r"\toprule",
         r" & Main & Excluding utilities \\", r"\midrule"]
for i, pollutant in enumerate(POLLUTANTS):
    if i:
        lines.append(r"\midrule")
    lines.append(r"\multicolumn{3}{l}{\textbf{Panel " + chr(65+i) + ": " + labels[pollutant] + r"}} \\")
    for statistic, values in regression_table.loc[pollutant].iterrows():
        label = statistic.replace("×", r"$\times$").replace("R²", r"$R^2$")
        lines.append(label + " & " + " & ".join(values) + r" \\")
lines.extend([r"\bottomrule", r"\end{tabular}", r"\par\smallskip",
    r"\begin{minipage}{0.95\linewidth}\footnotesize",
    r"\textit{Notes:} Outcome: " + ("natural log emissions" if LOG_OUTCOME else "emissions in tons") + ". "
    + "Years: " + ", ".join(map(str, YEARS)) + ". "
    + r"Unweighted state--sector--year OLS with sector and year fixed effects. "
    r"Post equals one from 2011 onward. Sector 7 is Other Industrial Processes, a broader category than cement. "
    r"Sectors 0 and 99 are excluded from both specifications; the alternative additionally excludes sector 1. "
    r"Missing/nonfinite outcomes are excluded; log models also exclude nonpositive emissions. "
    r"DC is treated as a state. State-clustered standard errors use a finite-sample correction and t-based p-values. "
    r"They do not account for arbitrary cross-state sector shocks; treatment occurs in a single sector. "
    r"* $p<0.10$, ** $p<0.05$, *** $p<0.01$.", r"\end{minipage}", r"\end{table}"])
OUTPUT.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Saved {OUTPUT} and cement_policy_regressions.csv")

## Event study by year relative to the policy

Replace the single treated × post term with a separate sector-7 interaction for each observed year except **2008**, the last observed pre-policy year:
\[
Y_{sjt}=\alpha_j+\lambda_t+
\sum_{k\ne2008}\beta_k\,1\{j=7\}1\{t=k\}+\varepsilon_{sjt}.
\]

Each coefficient is the change in the sector-7/control gap relative to 2008. The 2008 coefficient is normalized to zero and is not estimated. 

The x-axis shows calendar years relative to **2010**, the policy's effective year; 2011 is the first observed post-policy year. Intervals are pointwise 95% confidence intervals using cluster-adjusted t inference, not simultaneous confidence bands. 

The graph follows `LOG_OUTCOME` and saves a PNG.

In [ ]:
REFERENCE_YEAR = 2008
POLICY_YEAR = 2010
event_years = sorted(sample["year"].unique())
assert REFERENCE_YEAR in event_years
interaction_names = [f"treated_year_{year}" for year in event_years if year != REFERENCE_YEAR]
event_formula = "outcome ~ C(sector_code) + C(year) + " + " + ".join(interaction_names)
event_models, event_records = {}, []

for pollutant in POLLUTANTS:
    for specification, exclude_utilities in specifications.items():
        group = sample.loc[sample["pollutant"].eq(pollutant)].copy()
        if exclude_utilities:
            group = group.loc[group["sector_code"].ne(1)].copy()
        assert group.loc[group["sector_code"].eq(7), "year"].nunique() == len(event_years)
        for year in event_years:
            if year != REFERENCE_YEAR:
                group[f"treated_year_{year}"] = (group["sector_code"].eq(7) & group["year"].eq(year)).astype(int)
        model = smf.ols(event_formula, data=group, missing="raise").fit(
            cov_type="cluster",
            cov_kwds={"groups": group["state"], "use_correction": True, "df_correction": True},
            use_t=True,
        )
        assert np.linalg.matrix_rank(model.model.exog) == model.model.exog.shape[1]
        assert model.nobs == models[(pollutant, specification)].nobs
        event_models[(pollutant, specification)] = model
        intervals = model.conf_int(alpha=0.05)
        for year in event_years:
            reference = year == REFERENCE_YEAR
            term = f"treated_year_{year}"
            event_records.append({
                "pollutant": pollutant, "specification": specification,
                "year": year, "relative_year": year - POLICY_YEAR, "reference": reference,
                "coefficient": 0.0 if reference else model.params[term],
                "std_error": np.nan if reference else model.bse[term],
                "ci_lower": np.nan if reference else intervals.loc[term, 0],
                "ci_upper": np.nan if reference else intervals.loc[term, 1],
                "p_value": np.nan if reference else model.pvalues[term],
                "observations": int(model.nobs),
            })
event_results = pd.DataFrame(event_records)

In [ ]:
import matplotlib.pyplot as plt

plot_labels = {"NOX": r"NO$_x$", "SO2": r"SO$_2$", "PM25": r"PM$_{2.5}$"}
styles = {"Main": ("#0072B2", "o", -0.12), "Excluding utilities": ("#D55E00", "s", 0.12)}
fig, axes = plt.subplots(1, len(POLLUTANTS), figsize=(14, 6), sharex=True)
for ax, pollutant in zip(np.atleast_1d(axes), POLLUTANTS):
    ax.axhline(0, color="0.45", linewidth=0.9)
    ax.axvline(0, color="0.45", linestyle="--", linewidth=1)
    for specification, (color, marker, offset) in styles.items():
        series = event_results.loc[(event_results["pollutant"] == pollutant)
                                   & (event_results["specification"] == specification)].sort_values("year")
        estimated = series.loc[~series["reference"]]
        ax.errorbar(estimated["relative_year"] + offset, estimated["coefficient"],
                    yerr=[estimated["coefficient"] - estimated["ci_lower"],
                          estimated["ci_upper"] - estimated["coefficient"]],
                    fmt=marker, color=color, capsize=3, markersize=5, label=specification)
        # Open markers identify the omitted reference year; it has no confidence interval.
        ax.plot(REFERENCE_YEAR - POLICY_YEAR + offset, 0, marker=marker,
                markerfacecolor="white", markeredgecolor=color, linestyle="none", markersize=6)
    ax.set_title(plot_labels[pollutant])
    ax.set_xticks([year - POLICY_YEAR for year in event_years],
                  [f"{year - POLICY_YEAR:+d}\n({year})" for year in event_years])
    ax.set_xlabel("Years relative to 2010 (calendar year)")
    ax.set_ylabel("Effect relative to 2008 (log points)" if LOG_OUTCOME else "Effect relative to 2008 (tons)")
    ax.grid(axis="y", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)
fig.legend(*np.atleast_1d(axes)[0].get_legend_handles_labels(), loc="lower center",
           bbox_to_anchor=(0.5, 0.085), ncol=2, frameon=False, fontsize=13, markerscale=1.5,)
fig.tight_layout(rect=[0, 0.17, 1, 0.94])
fig.savefig("cement_policy_event_study.png", dpi=200, bbox_inches="tight")
plt.show()